In [2]:

# ============================================================
# CS 340 Project Two
# Grazioso Salvare Search and Rescue Dashboard
# Developed by Brian Minor
#
# This dashboard uses MongoDB and the Project One CRUD module
# to identify dogs that meet Grazioso Salvare's rescue criteria.
# ============================================================

# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------

from jupyter_dash import JupyterDash

import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output

import plotly.express as px
import pandas as pd
import base64
import re

from pathlib import Path

from ProjectOne_CRUD_Python_Module import AnimalShelter

JupyterDash.infer_jupyter_proxy_config()


# ------------------------------------------------------------
# 2. MongoDB Connection / Model
# ------------------------------------------------------------

# Credentials required for the private Codio assignment.
# Remove these before publishing code to a public repository.
username = "aacuser"
password = "SNHU1234"

# Connect through the CRUD Python module from Project One.
shelter = AnimalShelter(username, password)

# Retrieve the complete, unfiltered dataset.
df = pd.DataFrame.from_records(shelter.read({}))

# Remove MongoDB ObjectId values for Dash compatibility.
df.drop(columns=["_id"], errors="ignore", inplace=True)

print("MongoDB connection successful!")
print("Total records:", len(df))


# ------------------------------------------------------------
# 3. Rescue Criteria
# ------------------------------------------------------------

# Criteria from the official CS 340 Dashboard Specifications:
#
# Water Rescue:
#   Intact Female, 26-156 weeks
#
# Mountain or Wilderness:
#   Intact Male, 26-156 weeks
#
# Disaster or Individual Tracking:
#   Intact Male, 20-300 weeks

RESCUE_CRITERIA = {

    "water": {
        "breeds": [
            "Labrador Retriever Mix",
            "Chesapeake Bay Retriever",
            "Newfoundland"
        ],
        "sex": "Intact Female",
        "min_weeks": 26,
        "max_weeks": 156
    },

    "mountain": {
        "breeds": [
            "German Shepherd",
            "Alaskan Malamute",
            "Old English Sheepdog",
            "Siberian Husky",
            "Rottweiler"
        ],
        "sex": "Intact Male",
        "min_weeks": 26,
        "max_weeks": 156
    },

    "disaster": {
        "breeds": [
            "Doberman Pinscher",
            "German Shepherd",
            "Golden Retriever",
            "Bloodhound",
            "Rottweiler"
        ],
        "sex": "Intact Male",
        "min_weeks": 20,
        "max_weeks": 300
    }
}


# ------------------------------------------------------------
# 4. MongoDB Filtering Functions
# ------------------------------------------------------------

def build_rescue_query(rescue_type):
    """
    Build a MongoDB query based on the selected rescue type.

    The query filters by animal type, preferred sex,
    training age, and preferred breeds.
    """

    if rescue_type == "reset":
        return {}

    criteria = RESCUE_CRITERIA.get(rescue_type)

    if criteria is None:
        return {}

    # Match preferred breed names within breed descriptions.
    # This includes mixed-breed descriptions.
    breed_patterns = [
        {
            "breed": {
                "$regex": re.escape(breed),
                "$options": "i"
            }
        }
        for breed in criteria["breeds"]
    ]

    return {
        "animal_type": "Dog",

        "sex_upon_outcome": criteria["sex"],

        "age_upon_outcome_in_weeks": {
            "$gte": criteria["min_weeks"],
            "$lte": criteria["max_weeks"]
        },

        "$or": breed_patterns
    }


def retrieve_animals(rescue_type):
    """
    Retrieve filtered animal records using the
    Project One AnimalShelter CRUD module.
    """

    query = build_rescue_query(rescue_type)

    results = shelter.read(query)

    filtered_df = pd.DataFrame.from_records(results)

    filtered_df.drop(
        columns=["_id"],
        errors="ignore",
        inplace=True
    )

    if filtered_df.empty:
        filtered_df = pd.DataFrame(columns=df.columns)

    return filtered_df.to_dict("records")


# ------------------------------------------------------------
# 5. Load Grazioso Salvare Logo
# ------------------------------------------------------------

# Find the official logo in the current directory.
logo_files = [
    path for path in Path(".").glob("*.png")
    if "grazioso" in path.name.lower()
    or "salvare" in path.name.lower()
]

if logo_files:

    logo_path = logo_files[0]

    with open(logo_path, "rb") as image_file:
        encoded_image = base64.b64encode(
            image_file.read()
        ).decode("utf-8")

    # The specifications require the logo to link to SNHU.
    logo_component = html.A(
        href="https://www.snhu.edu",
        target="_blank",

        children=[
            html.Img(
                src="data:image/png;base64,{}".format(
                    encoded_image
                ),

                style={
                    "maxWidth": "260px",
                    "maxHeight": "140px",
                    "objectFit": "contain"
                }
            )
        ]
    )

    print("Logo loaded:", logo_path.name)

else:

    logo_component = html.Div(
        "Grazioso Salvare logo not found",
        style={"color": "red"}
    )

    print("WARNING: Official logo not found.")


# ------------------------------------------------------------
# 6. Dashboard Layout / View
# ------------------------------------------------------------

app = JupyterDash(__name__)

COLORS = {
    "navy": "#17324D",
    "blue": "#246A9C",
    "light": "#F3F7FA",
    "white": "#FFFFFF",
    "text": "#263746"
}

app.layout = html.Div(

    style={
        "fontFamily": "Arial, sans-serif",
        "backgroundColor": COLORS["light"],
        "padding": "25px",
        "color": COLORS["text"]
    },

    children=[

        # ----------------------------------------------------
        # Dashboard Header
        # ----------------------------------------------------

        html.Div(
            style={
                "textAlign": "center",
                "backgroundColor": COLORS["white"],
                "padding": "20px",
                "borderRadius": "10px",
                "marginBottom": "20px"
            },

            children=[

                logo_component,

                html.H1(
                    "Grazioso Salvare",
                    style={"color": COLORS["navy"]}
                ),

                html.H2(
                    "Search and Rescue Animal Dashboard",
                    style={
                        "color": COLORS["blue"],
                        "fontSize": "22px"
                    }
                ),

                html.H4(
                    "Developed by Brian Minor | CS 340 Project Two"
                ),

                html.P(
                    "Austin Animal Center Outcomes Database"
                )
            ]
        ),

        # ----------------------------------------------------
        # Rescue Filtering Controls
        # ----------------------------------------------------

        html.Div(
            style={
                "backgroundColor": COLORS["white"],
                "padding": "20px",
                "borderRadius": "10px",
                "marginBottom": "20px"
            },

            children=[

                html.H3("Rescue Type Filters"),

                html.P(
                    "Select a rescue category to display "
                    "qualifying animals."
                ),

                dcc.RadioItems(
                    id="filter-type",

                    options=[
                        {
                            "label": "Water Rescue",
                            "value": "water"
                        },
                        {
                            "label": "Mountain or Wilderness Rescue",
                            "value": "mountain"
                        },
                        {
                            "label": "Disaster or Individual Tracking",
                            "value": "disaster"
                        },
                        {
                            "label": "Reset - Show All Animals",
                            "value": "reset"
                        }
                    ],

                    value="reset",

                    labelStyle={
                        "display": "inline-block",
                        "marginRight": "25px",
                        "marginBottom": "10px",
                        "cursor": "pointer"
                    }
                )
            ]
        ),

        # ----------------------------------------------------
        # Results Summary
        # ----------------------------------------------------

        html.Div(
            id="results-summary",

            style={
                "fontSize": "17px",
                "fontWeight": "bold",
                "marginBottom": "15px"
            }
        ),

        # ----------------------------------------------------
        # Interactive Data Table
        # ----------------------------------------------------

        html.Div(
            style={
                "backgroundColor": COLORS["white"],
                "padding": "20px",
                "borderRadius": "10px",
                "marginBottom": "20px"
            },

            children=[

                html.H3("Animal Shelter Data"),

                dash_table.DataTable(

                    id="datatable-id",

                    columns=[
                        {
                            "name": column.replace(
                                "_", " "
                            ).title(),
                            "id": column,
                            "selectable": True
                        }
                        for column in df.columns
                    ],

                    data=df.to_dict("records"),

                    row_selectable="single",
                    selected_rows=[],

                    page_action="native",
                    page_current=0,
                    page_size=10,

                    sort_action="native",
                    sort_mode="multi",

                    filter_action="native",
                    column_selectable="single",

                    style_table={
                        "overflowX": "auto"
                    },

                    style_cell={
                        "textAlign": "left",
                        "minWidth": "110px",
                        "width": "150px",
                        "maxWidth": "250px",
                        "overflow": "hidden",
                        "textOverflow": "ellipsis",
                        "padding": "10px"
                    },

                    style_header={
                        "backgroundColor": COLORS["navy"],
                        "color": COLORS["white"],
                        "fontWeight": "bold",
                        "textAlign": "center"
                    },

                    style_data_conditional=[
                        {
                            "if": {"row_index": "odd"},
                            "backgroundColor": COLORS["light"]
                        }
                    ]
                )
            ]
        ),

        # ----------------------------------------------------
        # Breed Chart and Geolocation Map
        # ----------------------------------------------------

        html.Div(

            style={
                "display": "flex",
                "flexWrap": "wrap",
                "gap": "20px"
            },

            children=[

                # Breed distribution chart
                html.Div(
                    style={
                        "flex": "1 1 450px",
                        "minWidth": "300px",
                        "backgroundColor": COLORS["white"],
                        "padding": "15px",
                        "borderRadius": "10px"
                    },

                    children=[
                        html.H3("Breed Distribution"),
                        dcc.Graph(id="graph-id")
                    ]
                ),

                # Geolocation chart
                html.Div(
                    style={
                        "flex": "1 1 450px",
                        "minWidth": "300px",
                        "backgroundColor": COLORS["white"],
                        "padding": "15px",
                        "borderRadius": "10px"
                    },

                    children=[
                        html.H3("Animal Geolocation"),
                        html.Div(id="map-id")
                    ]
                )
            ]
        ),

        html.Hr(),

        html.Center(
            html.P(
                "CS 340 Project Two | Brian Minor | "
                "Grazioso Salvare"
            )
        )
    ]
)


# ------------------------------------------------------------
# 7. Controller: Rescue Filtering Callback
# ------------------------------------------------------------

@app.callback(
    [
        Output("datatable-id", "data"),
        Output("datatable-id", "selected_rows"),
        Output("datatable-id", "page_current"),
        Output("datatable-id", "filter_query"),
        Output("datatable-id", "sort_by"),
        Output("results-summary", "children")
    ],
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):

    # Retrieve filtered records from MongoDB.
    filtered_records = retrieve_animals(filter_type)

    count = len(filtered_records)

    labels = {
        "water": "Water Rescue",
        "mountain": "Mountain or Wilderness Rescue",
        "disaster": "Disaster or Individual Tracking",
        "reset": "All Shelter Animals"
    }

    selected_label = labels.get(
        filter_type,
        "All Shelter Animals"
    )

    summary = (
        f"{selected_label} | "
        f"Matching Records: {count:,}"
    )

    # Clear previous row selection, table search, and sorting.
    # This also restores the original table view on Reset.
    return filtered_records, [], 0, "", [], summary


# ------------------------------------------------------------
# 8. Controller: Breed Distribution Chart
# ------------------------------------------------------------

@app.callback(
    Output("graph-id", "figure"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(view_data):

    if view_data is None:
        chart_df = df.copy()
    else:
        chart_df = pd.DataFrame(view_data)

    if chart_df.empty or "breed" not in chart_df.columns:

        fig = px.bar(
            title="No Animals Match the Current Filters"
        )

        fig.add_annotation(
            text="No records available",
            x=0.5,
            y=0.5,
            xref="paper",
            yref="paper",
            showarrow=False
        )

        return fig

    breed_counts = (
        chart_df["breed"]
        .fillna("Unknown")
        .value_counts()
        .head(10)
        .reset_index()
    )

    breed_counts.columns = ["Breed", "Count"]

    fig = px.bar(
        breed_counts,
        x="Count",
        y="Breed",
        orientation="h",
        title="Top 10 Breeds in Current Results",
        labels={
            "Count": "Number of Animals",
            "Breed": "Breed"
        }
    )

    fig.update_layout(
        height=500,
        margin={
            "l": 20,
            "r": 20,
            "t": 60,
            "b": 40
        },
        yaxis={
            "categoryorder": "total ascending"
        }
    )

    return fig


# ------------------------------------------------------------
# 9. Controller: Geolocation Map
# ------------------------------------------------------------

@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows")
    ]
)
def update_map(view_data, selected_rows):

    if view_data is None:
        map_df = df.copy()
    else:
        map_df = pd.DataFrame(view_data)

    if map_df.empty:
        return html.Div(
            "No animal locations available.",
            style={
                "padding": "30px",
                "textAlign": "center"
            }
        )

    if not {
        "location_lat",
        "location_long"
    }.issubset(map_df.columns):

        return html.Div(
            "Location fields are unavailable."
        )

    # Select the first animal if no row is selected.
    if selected_rows and selected_rows[0] < len(map_df):
        row_index = selected_rows[0]
    else:
        row_index = 0

    selected_animal = map_df.iloc[row_index]

    latitude = pd.to_numeric(
        selected_animal.get("location_lat"),
        errors="coerce"
    )

    longitude = pd.to_numeric(
        selected_animal.get("location_long"),
        errors="coerce"
    )

    center = [30.2672, -97.7431]

    valid_coordinates = (
        pd.notna(latitude)
        and pd.notna(longitude)
        and -90 <= latitude <= 90
        and -180 <= longitude <= 180
    )

    markers = []

    if valid_coordinates:

        center = [
            float(latitude),
            float(longitude)
        ]

        animal_name = selected_animal.get("name", "")

        if (
            pd.isna(animal_name)
            or not str(animal_name).strip()
        ):
            animal_name = "Unnamed Animal"

        breed = str(
            selected_animal.get("breed", "Unknown")
        )

        animal_id = str(
            selected_animal.get("animal_id", "Unknown")
        )

        markers.append(
            dl.Marker(
                position=center,

                children=[
                    dl.Tooltip(breed),

                    dl.Popup([
                        html.H4(animal_name),
                        html.P("Animal ID: " + animal_id),
                        html.P("Breed: " + breed)
                    ])
                ]
            )
        )

    return dl.Map(
        style={
            "width": "100%",
            "height": "500px",
            "borderRadius": "8px"
        },

        center=center,
        zoom=10,

        children=[
            dl.TileLayer(),
            *markers
        ]
    )


# ------------------------------------------------------------
# 10. Controller: Highlight Selected Columns
# ------------------------------------------------------------

@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def update_styles(selected_columns):

    if selected_columns is None:
        selected_columns = []

    styles = [
        {
            "if": {"row_index": "odd"},
            "backgroundColor": COLORS["light"]
        }
    ]

    styles.extend([
        {
            "if": {"column_id": column},
            "backgroundColor": "#D2F3FF"
        }
        for column in selected_columns
    ])

    return styles


# ------------------------------------------------------------
# 11. Launch Dashboard
# ------------------------------------------------------------

print("Starting Grazioso Salvare Dashboard...")
print("Developed by Brian Minor - CS 340 Project Two")

app.run_server(
    mode="external",
    port=8060,
    debug=False
)


MongoDB connection successful!
Total records: 10000
Logo loaded: Grazioso Salvare Logo.png


 * Running on http://127.0.0.1:8060/ (Press CTRL+C to quit)
127.0.0.1 - - [14/Oct/2026 01:50:36] "GET /_alive_ac72caea-2d5c-4495-8c12-d026fde33f48 HTTP/1.1" 200 -


Starting Grazioso Salvare Dashboard...
Developed by Brian Minor - CS 340 Project Two
Dash app running on https://costumeepoxy-librapelican-3000.codio.io/proxy/8060/


127.0.0.1 - - [14/Oct/2026 01:50:41] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 01:50:41] "GET /_dash-dependencies HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 01:50:41] "GET /_dash-layout HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 01:50:42] "GET /_dash-component-suites/dash/dash_table/async-table.js HTTP/1.1" 304 -
127.0.0.1 - - [14/Oct/2026 01:50:42] "GET /_dash-component-suites/dash/dash_table/async-highlight.js HTTP/1.1" 304 -
127.0.0.1 - - [14/Oct/2026 01:50:42] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 01:50:42] "GET /_dash-component-suites/dash/dcc/async-graph.js HTTP/1.1" 304 -
127.0.0.1 - - [14/Oct/2026 01:50:42] "GET /_dash-component-suites/dash/dcc/async-plotlyjs.js HTTP/1.1" 304 -
127.0.0.1 - - [14/Oct/2026 01:50:42] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 01:50:42] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2026 01:50:43] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - -